In [ ]:
from ultralytics import YOLO
import cv2
import os

# -------------------------------------------------------
# 1. Load pretrained YOLO model
# -------------------------------------------------------
model = YOLO("yolo26n.pt")

# -------------------------------------------------------
# 2. Input video path
# -------------------------------------------------------
video_path = r"C:\Users\KG KA19SDF180254\Documents\cam\VID_20260813_022621-60fps.mp4"

# -------------------------------------------------------
# 3. Folder to save first 20 detected frames
# -------------------------------------------------------
output_folder = "detected_20_frames"

os.makedirs(output_folder, exist_ok=True)

# -------------------------------------------------------
# 4. Open video
# -------------------------------------------------------
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("ERROR: Could not open video.")
else:
    print("Video opened successfully.")

# -------------------------------------------------------
# 5. Process first 20 frames
# -------------------------------------------------------
frame_number = 0
max_frames = 20

while frame_number < max_frames:

    ret, frame = cap.read()

    if not ret:
        print("Video ended before 20 frames.")
        break

    frame_number += 1

    # ---------------------------------------------------
    # Object detection
    # ---------------------------------------------------
    results = model.predict(
        source=frame,
        conf=0.02,
        imgsz=640,
        verbose=False
    )

    result = results[0]

    # ---------------------------------------------------
    # Print detected objects
    # ---------------------------------------------------
    print(f"\nFrame {frame_number}")

    if len(result.boxes) == 0:
        print("No objects detected")

    else:
        for box in result.boxes:

            class_id = int(box.cls[0])
            confidence = float(box.conf[0])

            x1, y1, x2, y2 = box.xyxy[0].tolist()

            class_name = model.names[class_id]

            print(
                f"Object: {class_name} | "
                f"Confidence: {confidence:.2f} | "
                f"Box: ({x1:.0f}, {y1:.0f}, "
                f"{x2:.0f}, {y2:.0f})"
            )

    # ---------------------------------------------------
    # Draw bounding boxes
    # ---------------------------------------------------
    annotated_frame = result.plot()

    # ---------------------------------------------------
    # Save detected frame
    # ---------------------------------------------------
    output_path = os.path.join(
        output_folder,
        f"frame_{frame_number:02d}.jpg"
    )

    cv2.imwrite(output_path, annotated_frame)

    print("Saved:", output_path)

# -------------------------------------------------------
# 6. Release video
# -------------------------------------------------------
cap.release()

print("\n--------------------------------")
print("Processing completed.")
print(f"{frame_number} frames processed.")
print("Saved in:", output_folder)
print("--------------------------------")